## 4조 加油~~! 

### 데이터 전처리 입니다 ! 현재 raw 데이터 2개 파일이 있어요 



### 1. raw파일과 csv 파일 목록 확인 

In [1]:
from pathlib import Path

# 실제 프로젝트 폴더를 직접 지정
# 이 PROJECT_ROO 변수 안에 실제 프로젝트 폴더 경로를 저장  # 이건 내 컴퓨터 기준~ 
PROJECT_ROOT = Path(r"C:\dev\project\project_t4_V2")  

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 위치:", PROJECT_ROOT)
print("raw 폴더 위치:", RAW_DIR)
print()

if not RAW_DIR.exists():
    print("raw 폴더를 찾지 못했습니다.")
else:
    csv_files = sorted(RAW_DIR.glob("*.csv"))

    if not csv_files:
        print("raw 폴더에 CSV 파일이 없습니다.") 
    else:
        print("찾은 CSV 파일:")
        for number, file in enumerate(csv_files, start=1):       
            print(f"{number}. {file.name} / {file.stat().st_size:,} bytes")

프로젝트 위치: C:\dev\project\project_t4_V2
raw 폴더 위치: C:\dev\project\project_t4_V2\data\raw

찾은 CSV 파일:
1. 자동차제작결함신고정보.csv / 3,161,019 bytes
2. 차종별 리콜대수.csv / 3,999,666 bytes


In [2]:
import pandas as pd
from IPython.display import display

def read_csv_auto(file_path):
    # 한글 인코딩을 순서대로 시도합니다.
    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            data = pd.read_csv(
                file_path,
                encoding=encoding,
                dtype=str
            )
            return data, encoding
        except UnicodeDecodeError:
            continue
# 모든 인코딩을 시도했는데도 파일을 읽지 못했을 때
    raise ValueError("파일 인코딩을 찾지 못했습니다.")

# 파일별로 읽고 일부 행만 출력
dataframes = {}

for file_path in csv_files:
    data, used_encoding = read_csv_auto(file_path)
    dataframes[file_path.name] = data

    print("=" * 60)
    print("파일명:", file_path.name)
    print("사용 인코딩:", used_encoding)
    print("행 수:", len(data))
    print("열 수:", len(data.columns))
    print("열 이름:", data.columns.tolist())
    print()
    print("앞에서 5개 행:")
    
    display(data.head(5))

파일명: 자동차제작결함신고정보.csv
사용 인코딩: utf-8-sig
행 수: 64807
열 수: 4
열 이름: ['접수일자', '제작사', '차명', '모델년도']

앞에서 5개 행:


,접수일자,제작사,차명,모델년도
0,2019-01-02,메르세데스-벤츠,GLA45 AMG 4Matic,2015
1,2019-01-02,현대자동차,i40 Saloon,2012
2,2019-01-02,르노코리아,SM5,2008
3,2019-01-02,BMW,BMW 530i,2018
4,2019-01-02,BMW,BMW 520d,2016


파일명: 차종별 리콜대수.csv
사용 인코딩: cp949
행 수: 13560
열 수: 7
열 이름: ['제작자', '차명', '생산기간(부터)', '생산기간(까지)', '리콜개시일', '리콜대수', '리콜사유']

앞에서 5개 행:


,제작자,차명,생산기간(부터),생산기간(까지),리콜개시일,리콜대수,리콜사유
0,벤츠,ML280 CDI,2006-04-18,2009-03-18,2012-01-09,529,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
1,벤츠,E220 CDI,2006-07-04,2009-01-15,2012-01-09,717,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
2,벤츠,S350 BLUETEC,2010-08-05,2011-02-02,2012-01-09,75,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
3,벤츠,ML300 CDI,2009-07-15,2011-04-06,2012-01-09,554,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
4,벤츠,C220 CDI,2007-01-03,2008-12-15,2012-01-09,549,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...


## 1. raw 폴더 확인과 CSV 불러오기

리콜 파일은 한글 Windows 인코딩일 수 있으므로 여러 인코딩을 순서대로 시도합니다.

## 2. 열 이름 확인 

In [3]:
def read_csv_auto(path):
    # UTF-8과 CP949 중 실제 파일에 맞는 인코딩을 찾습니다.
    last_error = None
    for encoding in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeDecodeError as error:
            last_error = error
    raise last_error

def find_raw_file(keyword):
    matches = [p for p in csv_files if keyword in p.stem]
    if not matches:
        raise FileNotFoundError(f'원본 파일을 찾지 못했습니다: {keyword}')
    return matches[0]

defect_path = find_raw_file('자동차제작결함신고정보')
recall_path = find_raw_file('차종별 리콜대수')
defect_raw, defect_encoding = read_csv_auto(defect_path)
recall_raw, recall_encoding = read_csv_auto(recall_path)

print('신고 파일:', defect_path.name, defect_encoding, defect_raw.shape)
print('리콜 파일:', recall_path.name, recall_encoding, recall_raw.shape)
display(defect_raw.head())
display(recall_raw.head())

신고 파일: 자동차제작결함신고정보.csv utf-8-sig (64807, 4)
리콜 파일: 차종별 리콜대수.csv cp949 (13560, 7)


,접수일자,제작사,차명,모델년도
0,2019-01-02,메르세데스-벤츠,GLA45 AMG 4Matic,2015.0
1,2019-01-02,현대자동차,i40 Saloon,2012.0
2,2019-01-02,르노코리아,SM5,2008.0
3,2019-01-02,BMW,BMW 530i,2018.0
4,2019-01-02,BMW,BMW 520d,2016.0


,제작자,차명,생산기간(부터),생산기간(까지),리콜개시일,리콜대수,리콜사유
0,벤츠,ML280 CDI,2006-04-18,2009-03-18,2012-01-09,529,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
1,벤츠,E220 CDI,2006-07-04,2009-01-15,2012-01-09,717,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
2,벤츠,S350 BLUETEC,2010-08-05,2011-02-02,2012-01-09,75,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
3,벤츠,ML300 CDI,2009-07-15,2011-04-06,2012-01-09,554,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...
4,벤츠,C220 CDI,2007-01-03,2008-12-15,2012-01-09,549,디젤연료 내에 이물질을 걸러내 주는 장치(히터내장형 연료필터)에서 연료가 누유되는 ...


## 3. 열 이름과 원본행 정리



In [4]:
# 파일별 데이터 복사
defect_raw = dataframes["자동차제작결함신고정보.csv"].copy()
recall_raw = dataframes["차종별 리콜대수.csv"].copy()

# 열 이름 앞뒤 공백 제거 
# astype(str)은 값을 문자열(string) 형식으로 변환 ex: 2020>"2020"
defect_raw.columns = defect_raw.columns.astype(str).str.strip()
recall_raw.columns = recall_raw.columns.astype(str).str.strip()

print("신고 원본 열:", defect_raw.columns.tolist())
print("리콜 원본 열:", recall_raw.columns.tolist())

신고 원본 열: ['접수일자', '제작사', '차명', '모델년도']
리콜 원본 열: ['제작자', '차명', '생산기간(부터)', '생산기간(까지)', '리콜개시일', '리콜대수', '리콜사유']


In [5]:
### 열 이름 이해하기 쉬운 공통이름으로 바꾸기~ 한글의 저주~~

In [6]:
# 소비자 결함신고 데이터 열 이름 변경
# copy() 는 원본 데이터와 분리된 복사본 defect = 전처리할 복사본
defect = defect_raw.rename(columns={
    "접수일자": "received_date",
    "제작사": "manufacturer_name",
    "차명": "model_name",
    "모델년도": "model_year"
}).copy()

# 공식 리콜 데이터 열 이름 변경
recalls = recall_raw.rename(columns={
    "제작자": "manufacturer_name",
    "차명": "model_name",
    "생산기간(부터)": "production_start_date",
    "생산기간(까지)": "production_end_date",
    "리콜개시일": "recall_start_date",
    "리콜대수": "affected_count",
    "리콜사유": "recall_reason"
}).copy()
# tolist() pandas 열 이름 목록 객체 >> 일반 python 리스트 변환
print("신고 정리 후 열:", defect.columns.tolist())
print("리콜 정리 후 열:", recalls.columns.tolist())

신고 정리 후 열: ['received_date', 'manufacturer_name', 'model_name', 'model_year']
리콜 정리 후 열: ['manufacturer_name', 'model_name', 'production_start_date', 'production_end_date', 'recall_start_date', 'affected_count', 'recall_reason']


###
 아 빈행제거 안했다, 원본행 번호는 굳이 ? 딱히 중요한 데이터 아님 (문제가 생겼을 때 원본 파일의 몇 번째 행인지 다시 찾을 수 있음) 문제 안 생김

In [7]:
def clean_source(data):
    data = data.copy()

    # 열 이름 앞뒤 공백 제거
    data.columns = data.columns.astype(str).str.strip()

    # 공백만 있는 셀을 결측값으로 변경
    # r → 문자열을 그대로 해석, ^  → 문자열의 시작 \s  → 공백 문자 * → 앞의 공백이 0개 이상 $  → 문자열의 끝 == 공백 모두 찾아서 pd.NA 결측값으로 replace해라 
    # regex=True  → 규칙을 이용해 찾기 False면 ^\s*$라는 글자를 그대로 찾음
    data = data.replace(r"^\s*$", pd.NA, regex=True)

    # 모든 열이 비어 있는 행만 삭제
    data = data.dropna(how="all")

    # 전처리 후 행 번호를 새로 정리 = pandas 내부 인덱스를 다시 0부터 정리
    data = data.reset_index(drop=True)

    return data

# 앞 셀에서 이미 공통 영문 열 이름으로 바꾼 데이터에 결측·빈 행 정리를 적용합니다.
# 원본(defect_raw/recall_raw)을 다시 넣으면 manufacturer_name이 사라지므로 사용하지 않습니다.
defect = clean_source(defect)
recalls = clean_source(recalls)

print("정리 후 신고 열: ", defect.columns.tolist())
print("정리 후 리콜 열: ", recalls.columns.tolist())
assert "manufacturer_name" in defect.columns, "신고 데이터의 제조사 열 이름을 확인하세요."
assert "manufacturer_name" in recalls.columns, "리콜 데이터의 제조사 열 이름을 확인하세요."

정리 후 신고 열:  ['received_date', 'manufacturer_name', 'model_name', 'model_year']
정리 후 리콜 열:  ['manufacturer_name', 'model_name', 'production_start_date', 'production_end_date', 'recall_start_date', 'affected_count', 'recall_reason']


###
처음에는 패밀리카 제외하는 방식을 써서 했지만 데이터 질이 쓰레기라 다른 방법 
1단계: 대상 제조사 선택 .. 차명이 달라요 ex)현대자동차, 현대자동차(주)
2단계: 해당 제조사의 패밀리카 차종 선택
3단계: 선택된 차종만 신고·리콜 데이터에 남김

In [9]:
# 신고·리콜 데이터의 제조사 열을 합침.
manufacturer_series = pd.concat([
    defect["manufacturer_name"],
    recalls["manufacturer_name"]
])

# 제조사명 앞뒤 공백 정리
manufacturer_series = (
    manufacturer_series
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .dropna() # 아까한거 NA 결측 제거
)

# 제조사별 데이터 행 수 집계
manufacturer_list = (
    manufacturer_series
    .value_counts()
    .rename_axis("raw_manufacturer_name")
    .reset_index(name="row_count")
)

print("고유 제조사 수:", len(manufacturer_list))

print("신고 데이터 상위 15개")
display(defect["manufacturer_name"].value_counts().head(15))

print("리콜 데이터 상위 15개")
display(recalls["manufacturer_name"].value_counts().head(15))

고유 제조사 수: 342
신고 데이터 상위 15개


manufacturer_name
현대자동차           19342
기아              14786
한국지엠             4653
르노코리아            3748
BMW              2853
메르세데스-벤츠         2434
폭스바겐그룹           2056
아우디폭스바겐코리아       2027
KG 모빌리티          1716
재규어랜드로버코리아       1307
혼다코리아            1249
르노코리아자동차          960
한국토요타자동차          726
테슬라코리아            679
지엠대우오토앤테크놀로지      678
Name: count, dtype: int64

리콜 데이터 상위 15개


manufacturer_name
비엠더블유        4409
벤츠           2559
폭스바겐그룹       1002
현대자동차         417
만트럭버스         416
포르쉐           344
포드            325
한불모터스         324
재규어랜드로버       323
토요타           273
기아            263
스텔란티스         242
비엠더블유(이륜)     225
에프엠케이         199
볼보            193
Name: count, dtype: int64

### 
차량 회사 이름 통일하고 11개 차량 회사 추출해서 거기서 패밀리카 차량 뽑기 .. 패밀리카를 선별할수 있는데이터가 없어서 차명으로 추출하는 방법 밖에 없어

In [ ]:
import re
import unicodedata
import pandas as pd

# --------------------------------------------------
# 1. 제조사명 정리입니다
# --------------------------------------------------
# 표기를 단순하게 만드는 함수
def normalize_key(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value)).casefold()
    text = re.sub(r"[^0-9a-z가-힣]+", "", text)  # 한글, 영문, 숫자를 제외한 공백·괄호·하이픈 등을 제거
    # 메르세데스-벤츠,메르세데스 벤츠,메르세데스벤츠 전부 메르세데스벤츠로 바뀜

    return text


# 표준 제조사명: 원본에서 발견될 수 있는 이름
MANUFACTURER_ALIASES = {
    "KG 모빌리티": [
        "KG모빌리티",
        "케이지모빌리티",
        "KG Mobility"
    ],
    "BMW": [
        "BMW",
        "비엠더블유"
    ],
    "기아": [
        "기아",
        "기아 주식회사",
        "KIA"
    ],
    "르노코리아": [
        "르노코리아",
        "르노코리아자동차",
        "르노삼성자동차",
        "Renault Korea"
    ],
    "메르세데스 벤츠": [
        "메르세데스-벤츠",
        "메르세데스 벤츠",
        "메르세데스벤츠",
        "벤츠",
        "Mercedes-Benz"
    ],
    "볼보": [
        "볼보",
        "볼보자동차코리아",
        "Volvo"
    ],
    "토요타": [
        "토요타",
        "한국토요타자동차",
        "도요타",
        "Toyota"
    ],
    "재규어랜드로버": [
        "재규어랜드로버",
        "재규어 랜드로버",
        "재규어랜드로버코리아",
        "Jaguar Land Rover"
    ],
    "포드": [
        "포드",
        "포드세일즈서비스코리아",
        "Ford"
    ],
    "현대자동차": [
        "현대자동차",
        "현대자동차(주)",
        "Hyundai"
    ],
    "혼다코리아": [
        "혼다",
        "혼다코리아",
        "Honda"
    ]
}
# 원본 제조사명 alias도 비교하기 쉽게 미리 정리

NORMALIZED_ALIASES = {
    standard_name: [
        normalize_key(alias)
        for alias in aliases
    ]
    for standard_name, aliases in MANUFACTURER_ALIASES.items()
}
# 원본 제조사명을 받아서 표준 제조사명
def standardize_manufacturer(value):
    value_key = normalize_key(value)

    if not value_key:
        return pd.NA

    for standard_name, aliases in NORMALIZED_ALIASES.items(): # 표준 제조사별 alias 목록을 하나씩 확인합니다.
        if any(alias in value_key for alias in aliases): # alias 중 하나라도 포함되면 표준 제조사명을 반환
            return standard_name

    return pd.NA


# 제조사명을 표준 이름으로 변경
defect["manufacturer_standard"] = (
    defect["manufacturer_name"]
    .map(standardize_manufacturer)
)

recalls["manufacturer_standard"] = (
    recalls["manufacturer_name"]
    .map(standardize_manufacturer)
)


# 11개 제조사만 추출
defect_selected = defect[
    defect["manufacturer_standard"].notna()
].copy()

recalls_selected = recalls[
    recalls["manufacturer_standard"].notna()
].copy()


# 조회에 사용할 제조사 열을 표준 이름으로 교체
defect_selected["manufacturer_name"] = (
    defect_selected["manufacturer_standard"]
)

recalls_selected["manufacturer_name"] = (
    recalls_selected["manufacturer_standard"]
)

defect_selected.drop(
    columns=["manufacturer_standard"],
    inplace=True # 데이터프레임 자체를 바로 수정
)

recalls_selected.drop(
    columns=["manufacturer_standard"],
    inplace=True
)


print("제조사 통일 후 신고 데이터:", defect_selected.shape)
print("제조사 통일 후 리콜 데이터:", recalls_selected.shape)

display(
    defect_selected["manufacturer_name"]
    .value_counts()
)

display(
    recalls_selected["manufacturer_name"]
    .value_counts()
)

제조사 통일 후 신고 데이터: (50609, 4)
제조사 통일 후 리콜 데이터: (6838, 7)


manufacturer_name
현대자동차       19342
기아          14786
르노코리아        4708
BMW          2858
메르세데스 벤츠     2434
KG 모빌리티      2259
재규어랜드로버      1307
혼다코리아        1249
토요타           726
포드            509
볼보            431
Name: count, dtype: int64

manufacturer_name
BMW        4634
현대자동차       417
포드          325
재규어랜드로버     323
토요타         273
볼보          270
기아          263
혼다코리아       237
르노코리아        66
KG 모빌리티      30
Name: count, dtype: int64

### 패밀리카 추출

In [11]:
# 패밀리카로 볼 MPV·미니밴
MPV_PATTERNS = [
    "카니발", "carnival",
    "스타리아", "staria",
    "스타렉스", "starex",
    "시에나", "sienna",
    "알파드", "alphard",
    "오딧세이", "odyssey",
    "카렌스", "carens"
]

# 패밀리카로 볼 SUV
SUV_PATTERNS = [
    "싼타페", "santa fe",
    "쏘렌토", "sorento",
    "투싼", "tucson",
    "스포티지", "sportage",
    "팰리세이드", "palisade",
    "모하비", "mohave",
    "니로", "niro",
    "코나", "kona",
    "투아렉", "touareg",
    "티구안", "tiguan",
    "렉스턴", "rexton",
    "티볼리", "tivoli",
    "토레스", "torres",
    "rav4", "cr-v",
    "x5", "x7",
    "gle", "gls",
    "gla", "glb", "glc", "glk",
    "eqa", "eqb", "eqc",
    "ml250", "ml280", "ml300", "ml350",
    "qm3", "qm5", "qm6",
    "xm3", "arkana", "koleos", "captur",
    "e-pace", "f-pace", "i-pace",
    "discovery", "range rover", "defender",
    "evoque", "freelander", "프리랜더",
    "xc90", "explorer",
    "gv70", "gv80"
]

MPV_KEYS = [
    normalize_key(pattern)
    for pattern in MPV_PATTERNS
]

SUV_KEYS = [
    normalize_key(pattern)
    for pattern in SUV_PATTERNS
]


def classify_family_car(model_name):
    model_key = normalize_key(model_name)

    if any(pattern in model_key for pattern in MPV_KEYS):
        return "MPV/미니밴"

    if any(pattern in model_key for pattern in SUV_KEYS):
        return "SUV"

    return pd.NA

### 파일로 저장하기전에 DATA 확인

In [12]:
# 제조사 11개 데이터에 차종 분류 적용
defect_selected["vehicle_type"] = (
    defect_selected["model_name"]
    .map(classify_family_car)
)

recalls_selected["vehicle_type"] = (
    recalls_selected["model_name"]
    .map(classify_family_car)
)

### 1. 제조사별 데이터 확인

In [13]:
print("신고 데이터 제조사")
display(
    defect_selected["manufacturer_name"]
    .value_counts()
)

print("리콜 데이터 제조사")
display(
    recalls_selected["manufacturer_name"]
    .value_counts()
)

신고 데이터 제조사


manufacturer_name
현대자동차       19342
기아          14786
르노코리아        4708
BMW          2858
메르세데스 벤츠     2434
KG 모빌리티      2259
재규어랜드로버      1307
혼다코리아        1249
토요타           726
포드            509
볼보            431
Name: count, dtype: int64

리콜 데이터 제조사


manufacturer_name
BMW        4634
현대자동차       417
포드          325
재규어랜드로버     323
토요타         273
볼보          270
기아          263
혼다코리아       237
르노코리아        66
KG 모빌리티      30
Name: count, dtype: int64

### 2. SUV·MPV 분류 건수 확인

In [14]:
print("신고 데이터 차종 분류")
display(
    defect_selected["vehicle_type"]
    .value_counts(dropna=False)
)

print("리콜 데이터 차종 분류")
display(
    recalls_selected["vehicle_type"]
    .value_counts(dropna=False)
)

신고 데이터 차종 분류


vehicle_type
NaN        34872
SUV        12835
MPV/미니밴     2902
Name: count, dtype: int64

리콜 데이터 차종 분류


vehicle_type
NaN        6027
SUV         693
MPV/미니밴     118
Name: count, dtype: int64

### 3. 실제 차명 확인

In [15]:
display(
    defect_selected[
        ["manufacturer_name", "model_name", "vehicle_type"]
    ]
    .drop_duplicates()
    .sort_values(
        ["manufacturer_name", "vehicle_type", "model_name"]
    )
    .head(10)
)
display(
    recalls_selected[
        ["manufacturer_name", "model_name", "vehicle_type"]
    ]
    .drop_duplicates()
    .sort_values(
        ["manufacturer_name", "vehicle_type", "model_name"]
    )
    .head(10)
)

,manufacturer_name,model_name,vehicle_type
15370,BMW,BMW X5 3.0,SUV
3993,BMW,BMW X5 3.0d,SUV
29428,BMW,BMW X5 3.0si,SUV
7883,BMW,BMW X5 M50d,SUV
230,BMW,BMW X5 xDrive30d,SUV
27925,BMW,BMW X5 xDrive30d M Sport Package,SUV
38958,BMW,BMW X5 xDrive30d xLine,SUV
2902,BMW,BMW X5 xDrive35i,SUV
2441,BMW,BMW X5 xDrive40d,SUV
48424,BMW,BMW X5 xDrive40e iPerformance,SUV


,manufacturer_name,model_name,vehicle_type
7044,BMW,BMW X5,SUV
10855,BMW,BMW X5 3.0si,SUV
4328,BMW,BMW X5 M,SUV
11037,BMW,BMW X5 M Competition,SUV
3783,BMW,BMW X5 M50d,SUV
6456,BMW,BMW X5 M50i,SUV
11038,BMW,BMW X5 M60i xDrive,SUV
11958,BMW,BMW X5 sDrive40i,SUV
9435,BMW,BMW X5 xDrive 30d,SUV
4159,BMW,BMW X5 xDrive 35d,SUV


### 최종추출

In [16]:
defect_family = defect_selected[
    defect_selected["vehicle_type"].notna()
].copy()

recalls_family = recalls_selected[
    recalls_selected["vehicle_type"].notna()
].copy()

### 이제 전처리 데이터 파일로 만들기

In [17]:
from pathlib import Path

# 전처리 결과 저장 폴더
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "processed_data"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 패밀리카만 남김
defect_family = defect_selected[
    defect_selected["vehicle_type"].notna()
].copy()

recalls_family = recalls_selected[
    recalls_selected["vehicle_type"].notna()
].copy()

# 파일 경로
defect_output = (
    OUTPUT_DIR
    / "자동차제작결함신고정보_11개제조사_패밀리카.csv"
)

recall_output = (
    OUTPUT_DIR
    / "차종별리콜대수_11개제조사_패밀리카.csv"
)

# CSV 저장
defect_family.to_csv(
    defect_output,
    index=False,
    encoding="utf-8-sig"
)

recalls_family.to_csv(
    recall_output,
    index=False,
    encoding="utf-8-sig"
)

print("신고 파일 저장 완료:", defect_output)
print("리콜 파일 저장 완료:", recall_output)
print("신고 데이터 행 수:", len(defect_family))
print("리콜 데이터 행 수:", len(recalls_family))

신고 파일 저장 완료: C:\dev\project\project_t4_V2\data\processed\processed_data\자동차제작결함신고정보_11개제조사_패밀리카.csv
리콜 파일 저장 완료: C:\dev\project\project_t4_V2\data\processed\processed_data\차종별리콜대수_11개제조사_패밀리카.csv
신고 데이터 행 수: 15737
리콜 데이터 행 수: 811


In [19]:
print("=== 최종 전처리 결과 확인 ===")

print("신고 데이터 행 수:", len(defect_family))
print(
    "신고 데이터 제조사 수:",
    defect_family["manufacturer_name"].nunique()
)

print("리콜 데이터 행 수:", len(recalls_family))
print(
    "리콜 데이터 제조사 수:",
    recalls_family["manufacturer_name"].nunique()
)

print("\n신고 데이터 제조사별 행 수")
display(
    defect_family["manufacturer_name"]
    .value_counts()
)

print("\n리콜 데이터 제조사별 행 수")
display(
    recalls_family["manufacturer_name"]
    .value_counts()
)

=== 최종 전처리 결과 확인 ===
신고 데이터 행 수: 18492
신고 데이터 제조사 수: 11
리콜 데이터 행 수: 1516
리콜 데이터 제조사 수: 11

신고 데이터 제조사별 행 수


manufacturer_name
기아          7775
현대자동차       6105
르노코리아       2185
KG 모빌리티     1122
메르세데스 벤츠     528
혼다코리아        304
포드           162
BMW          137
재규어랜드로버      106
볼보            35
토요타           33
Name: count, dtype: int64


리콜 데이터 제조사별 행 수


manufacturer_name
메르세데스 벤츠    636
BMW         349
기아          131
현대자동차       131
혼다코리아        61
토요타          55
포드           48
재규어랜드로버      46
볼보           24
르노코리아        23
KG 모빌리티      12
Name: count, dtype: int64

```text
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⣴⣶⣶⣶⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⣿⣿⣿⣿⣿⣿⣷⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣼⣿⣿⣿⣿⣿⣿⣿⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⣿⣿⣿⣿⣿⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢿⣿⣿⣿⣿⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣿⣿⣿⣿⣿⣾⡋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠻⠿⠻⠿⠻⢤⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡀⢀⢤⣒⣽⢍⣸⣷⣶⣉⣁⣀⠑⠤⠤⢀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⢎⣨⡟⣉⣄⣴⡎⣳⡏⡏⣡⡦⣼⠋⣷⢦⣠⣫⠳⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣰⢿⣅⠈⢹⠟⠸⡟⠓⢟⡷⠥⢷⢿⢻⢾⠇⣴⢮⡏⠆⢹⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡸⣾⣟⢬⣍⣊⠠⡐⢶⠨⢀⢠⢡⢸⠀⠸⢸⠀⢠⢸⣶⡳⡿⣻⣃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⡎⣧⡾⣿⣏⢯⢳⢶⣿⡟⢿⠟⣿⣾⢠⣮⣘⣴⠎⣬⣾⣯⣮⡞⢯⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣾⢯⣍⣶⡽⣿⡿⢦⠻⢸⢘⣿⣿⣾⣿⣿⣿⡏⢿⠬⢿⡸⢻⣛⣹⡴⢮⣻⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⣴⠟⣜⣿⣿⢿⡽⠋⠀⠘⣆⠤⡧⡃⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⠶⣿⣿⣋⣩⣶⣿⣷⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⣾⠟⣌⢦⣿⢡⡿⠊⠀⠀⠀⠀⠸⣆⡧⣈⢷⣤⣤⢲⣶⠀⠀⠀⢀⣿⣰⣷⢇⢻⣥⣿⣻⣮⢿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⡛⣡⣿⣿⣿⡿⠋⠀⠀⠀⠀⠀⠀⠀⢻⣿⣻⣾⣿⣧⣿⡏⠀⠀⢀⣾⣷⣿⡯⣿⡀⠹⣿⣧⡈⢷⣿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⢀⣴⣟⣽⣿⣿⣿⠟⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠳⣻⡿⢟⣓⡿⠀⠀⠀⢸⣽⣾⣿⢿⣹⣵⠀⠘⢷⣮⡱⣿⣮⣿⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⢀⣴⣿⢻⣽⣿⠿⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠀⠙⠶⣶⣿⠇⠀⠀⢀⣯⣾⣯⣯⣈⡇⢷⣧⠀⠀⠙⠻⣾⣿⡻⣿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⢀⣴⣶⡿⣯⢿⡿⠚⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⡶⡒⣾⣿⠃⠀⠀⠀⣼⣟⣍⢳⡻⡯⣯⣩⣯⡆⠀⠀⠀⠈⠻⣝⢫⣻⣿⢷⣦⣠⠤⣴⡶⠶⠀⠀⠀⠀⠀
⠀⣰⣿⣿⣿⣿⠞⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⢿⠀⣿⠃⠀⠀⠀⣜⢯⣹⣯⡏⣟⣿⡧⠿⢧⠁⠀⠀⠀⠀⠀⠈⠙⢾⣿⣟⣭⣥⣔⣧⣀⠀⠀⠀⠀⠀⠀
⣸⣿⡿⣿⠏⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡘⠻⡾⣿⡀⠀⠀⣰⣿⣺⣟⣯⠟⢥⡿⢟⣷⣮⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣷⣻⣯⣿⡻⠿⣦⡄⠀⠀⠀
⠻⠿⢱⠟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠦⣵⣿⣿⡻⣿⣿⣷⣯⡾⢷⣯⣏⣣⣶⣧⢷⣮⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣻⣿⣶⣄⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢺⣶⣿⣷⣾⣿⣿⣿⣿⢿⣿⡿⣿⣿⣿⣞⣶⠿⣽⣽⣿⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠇⠙⠋⠁⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⡗⢿⣿⣧⡷⣿⣿⣿⣷⣿⣿⣿⣿⣿⣿⣿⣽⣿⣯⣵⢹⣷⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣿⣇⣷⡿⣿⣿⣿⣮⣿⣿⣿⣿⣿⣿⣿⣿⣟⡫⣻⡿⣷⣾⣻⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣿⣿⣿⣿⣻⣷⣿⣿⣯⣿⣿⣾⣿⣿⢿⣿⣶⣾⣏⣼⣞⣿⣿⣥⢿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣾⣿⣯⣿⣿⣷⣿⣷⣿⡭⡿⣽⠏⠉⠉⠉⠲⣤⣽⣼⣿⣿⣥⣧⣧⣦⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⢿⡷⣓⣿⣿⣿⣻⣿⣻⣻⣯⠋⠀⠀⠀⠀⠀⢹⣾⣟⣿⣿⣿⣿⡿⡛⢻⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣸⠀⠘⣿⣿⣭⡽⠟⠛⠛⠛⠋⠀⠀⠀⠀⠀⠀⢘⡟⣿⣿⣏⣍⣖⣹⢿⡿⡞⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣴⠋⠀⢈⣿⣿⡟⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⠛⠛⠻⡉⠉⠀⢈⡖⢸⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣼⣥⣀⠠⣿⣿⠏⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢳⣀⣴⣶⣷⣶⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠗⠈⠑⢮⣿⡾⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⡟⠋⠁⠘⡌⠙⢆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣼⢋⠀⠄⢱⡿⡟⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢩⠁⠀⢛⣱⡀⠈⢆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡮⣤⣦⡀⢀⡾⡝⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⡆⠖⠋⠁⢣⠀⠈⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣾⠃⡰⠉⣳⣾⡝⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡾⠋⠠⠈⣆⠀⠸⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡜⠀⠀⢠⣾⡿⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠣⡀⠀⠈⠀⠀⢱⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⢁⢤⣰⣿⠟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣄⠀⠀⠀⠀⢇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠃⣮⣿⡿⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢆⠀⠀⣀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⢤⡿⡟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢣⢊⠬⠆⠑⣄⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⣰⣿⣿⡿⣥⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢦⣶⡒⢶⡞⠢⣄⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⣠⣾⣛⡏⡻⣳⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⢿⣿⣿⣾⢷⣿⣿⣶⣯⡿⢆⡀
⠀⠀⠀⠀⠀⠀⠉⠉⠚⠚⠛⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠦⠿⠿⠿⠿⠿⠿⠿⠿⠷⠷⠾
'''